# Aula 4 — Notebook 1

## Do protótipo a um workflow multiagente operacional

Nesta etapa, o projeto passa a usar **quatro agentes reais**, coordenados por LangGraph:

1. agente de triagem;
2. agente investigador;
3. agente revisor;
4. agente planejador de ação.

O objetivo pedagógico não é afirmar que quatro agentes são necessariamente melhores do que um. A arquitetura multiagente foi escolhida porque permite observar, separadamente, decisões de triagem, investigação, crítica e planejamento. Essa separação aumenta a visibilidade das responsabilidades, mas também acrescenta custo, latência, pontos de falha e complexidade operacional.

Ao final do notebook, compare essa solução com a alternativa de agente único apresentada em `target_project/incident_agent/app/single_agent_baseline.py`.


## 1. Por que LangGraph nesta etapa?

LangGraph representa o workflow como um grafo de estados. Cada nó recebe o estado atual e devolve apenas as alterações produzidas naquela etapa.

No projeto, os nós agênticos realmente chamam um LLM. Os nós determinísticos executam tarefas que devem permanecer previsíveis, como validação de contratos, aplicação de políticas, execução de ferramentas permitidas, contabilização de métricas e persistência.

A distinção é deliberada:

- **agentes** interpretam contexto e propõem decisões;
- **código** impõe limites, valida e executa apenas operações autorizadas;
- **LangGraph** coordena a ordem e as rotas do processo.

O grafo principal é:

```text
triage_agent
     ↓
execute_tools
     ↓
investigation_agent
     ↓
deterministic_validation
     ↓
review_agent
  ↙      ↘
retry   action_planner_agent
            ↓
       policy_check
         ↙     ↘
 human_review  simulation
```


In [ ]:
from pathlib import Path
import json
import sys

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
INCIDENT_AGENT_DIR = PROJECT_ROOT / "target_project/incident_agent"
sys.path.insert(0, str(PROJECT_ROOT))
sys.path.insert(0, str(INCIDENT_AGENT_DIR))

from shared.aula_4.langgraph_helpers import build_initial_state
from shared.aula_4.validation import validate_incident_input


## 2. O estado como contrato de execução

O estado não é a memória interna de um objeto. É um dicionário explícito, tipado por `WorkflowState`, que registra o que cada etapa produziu.

Entre os campos principais estão:

- `incident`: entrada original;
- `triage`: classificação e escolha de ferramentas do primeiro agente;
- `selected_tools` e `search_queries`: decisões que o código poderá executar;
- `retrieved_evidence`: evidências retornadas pelas ferramentas;
- `investigation`: hipóteses e recomendações do investigador;
- `deterministic_validation`: verificações independentes do LLM;
- `review`: avaliação crítica do agente revisor;
- `proposed_action`: ação estruturada pelo planejador;
- `metrics` e `events`: rastreabilidade operacional.

Esse desenho facilita depuração, testes, retries e comparação entre execuções.


In [ ]:
incident_path = INCIDENT_AGENT_DIR / "data/incidents/incident_documented.json"
incident_payload = json.loads(incident_path.read_text(encoding="utf-8"))
incident = validate_incident_input(incident_payload)

initial_state = build_initial_state(incident)
initial_state


## 3. Funções de domínio, agentes e adaptadores do framework

As funções em `shared/aula_4` foram separadas para não dependerem do grafo sempre que possível.

Exemplos independentes do framework:

- validação do incidente;
- busca lexical em documentos;
- aplicação da política de ações;
- criação de eventos e métricas.

Exemplos específicos de LangGraph:

- criação do `StateGraph`;
- declaração de nós;
- arestas condicionais;
- compilação e invocação do workflow.

Observe também que o agente de triagem **escolhe** as ferramentas, mas não executa código arbitrário. A função `execute_selected_tools` aceita somente nomes presentes em uma lista controlada.


In [ ]:
from shared.aula_4.retrieval import (
    AVAILABLE_TOOLS,
    execute_selected_tools,
    load_json_documents,
)

knowledge_dir = INCIDENT_AGENT_DIR / "data/knowledge"
documents = load_json_documents(knowledge_dir)

# Simulação da decisão que, no workflow real, é produzida pelo agente de triagem.
selected_tools = ["search_runbooks", "search_logs"]
search_queries = ["timeout raw_orders orders_daily"]

evidence = execute_selected_tools(selected_tools, search_queries, documents)
evidence


## 4. O grafo completo e as chamadas aos agentes

Cada agente é criado em `shared/aula_4/llm.py` por meio de `with_structured_output`. A chamada real ao modelo ocorre dentro de `invoke_structured_agent`, usada pelos nós:

- `triage_node`;
- `investigate_node`;
- `review_node`;
- `action_planner_node`.

Portanto, LangGraph não é o agente. Ele decide **quando** cada função será executada. A função do nó chama o modelo e grava o resultado no estado.

As validações e políticas continuam fora do LLM para que uma resposta persuasiva do modelo não possa contornar regras do sistema.


In [ ]:
from shared.aula_4.config import get_settings
from app.workflow import build_workflow

# Esta célula exige uma OPENAI_API_KEY configurada no ambiente ou em target_project/incident_agent/.env.
# settings = get_settings(INCIDENT_AGENT_DIR / ".env")
# workflow = build_workflow(settings, knowledge_dir)
# workflow

## 5. Executando o workflow

A execução exige uma chave válida do provedor configurado. Antes de remover os comentários da célula, copie `.env.example` para `.env` e defina `OPENAI_API_KEY`.

Durante a execução, observe:

1. quais ferramentas a triagem selecionou;
2. quais evidências foram retornadas;
3. se o revisor aprovou ou solicitou revisão;
4. quantas chamadas ao modelo ocorreram;
5. se a política permitiu simulação ou exigiu intervenção humana.


In [ ]:
from app.workflow import run_workflow

# settings = get_settings(INCIDENT_AGENT_DIR / ".env")
# result = run_workflow(incident, settings, knowledge_dir)
# result

## 6. Exercício orientado — multiagente ou agente único?

Execute o mesmo incidente com o workflow multiagente e com `run_single_agent_baseline`. Compare:

- qualidade e consistência da análise;
- quantidade de tokens;
- latência;
- facilidade de entender onde ocorreu um erro;
- número de componentes que precisam ser mantidos;
- necessidade real de revisão independente.

A conclusão esperada não é “multiagentes são melhores”. Em tarefas simples, um agente único com bom contrato e validações pode ser mais barato, rápido e suficiente. A arquitetura multiagente se justifica quando a separação de responsabilidades, a revisão independente ou o controle de rotas produz ganhos que compensam a complexidade adicional.
